Imports:

In [8]:
import torch
import transformers

## Part A

Getting the model (`DistilGPT2`)...

In [9]:
model_name = "distilbert/distilgpt2"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name, device_map='auto')
model = transformers.AutoModelForCausalLM.from_pretrained(model_name, device_map='auto')

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Then the paragraph I'll use is below (from _The Martian_)

In [10]:
paragraph = r"""Teddy swiveled his chair and looked out the window to the sky beyond. Night was edging in. "What must it be like?” He pondered. "He's stuck out there. He thinks he's totally alone and that we all gave up on him. What kind of effect does that have on a man's psychology?" He turned back to Venkat. "I wonder what he's thinking right now." LOG ENTRY: SOL 61 How come Aquaman can control whales? They're mammals! Makes no sense"""

Then the perplexity of the sequence is 

In [20]:
tokens = tokenizer(paragraph, return_tensors="pt").to(model.device)
with torch.inference_mode():
    h = model(**tokens, labels=tokens.input_ids).loss
    ppl = torch.e ** h.item()
    display(ppl)

49.106479599645056

With the perplexity of a shuffled version of it

In [12]:
torch.manual_seed(42)
with torch.inference_mode():
    shuffled_tokens = tokenizer(paragraph, return_tensors="pt").to(model.device)
    n = shuffled_tokens["input_ids"].shape[-1]
    shuffled_tokens["input_ids"] = shuffled_tokens["input_ids"][..., torch.randperm(n)]
    ppl = 2 ** model(**shuffled_tokens, labels=shuffled_tokens["input_ids"]).loss.item()
ppl

521.4959165909193

This increase makes sense since the second one is essentially asking the model to predict outputs for a random sequence, whereas the original is asking the model to predict from text. The model is much more surprised by the tokens in the random sequence vs. that of an English passage.

## Part B

Below is the output of greedy decoding:

In [67]:
prompt = "Once upon a time"
tokens = tokenizer(prompt, return_tensors="pt").to(model.device)

torch.manual_seed(0)
output = model.generate(**tokens, max_length=150, temperature=0.3)
print(tokenizer.decode(output[0]))

Once upon a time of war, the United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence. The United States was the only country in the world to have a military presence


Then with sampling and the temperature options ($T = 0$ is excluded since that is an invalid value for temperature, and is the same as greedy decoding):

In [68]:
import numpy as np
from transformers import GenerationConfig


prompt = "Once upon a time"
tokens = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    for i in range(1, 6):
        temp = i*0.3
        output = model.generate(
            **tokens, max_length=150, temperature=temp, do_sample=True
        )
        print(f"Temperature: {temp}\n{tokenizer.decode(output[0])}\n")

Temperature: 0.3
Once upon a time of war, the United States began to develop a system of military force that would have been used to defend the United States. The United States was a major power in the world.


The United States was a major power in the world.
The United States was a major power in the world.
The United States was a major power in the world.
The United States was a major power in the world.
The United States was a major power in the world.
The United States was a major power in the world.
The United States was a major power in the world.
The United States was a major power in the world.
The United States was a major power in the world.

Temperature: 0.6
Once upon a time when the world was still a place of war, the world was still a place of peace.


The war was already over, but it was time to start the war against the other factions.
The fight had begun, but it was time to stop the war.<|endoftext|>

Temperature: 0.8999999999999999
Once upon a time of war, but before 

The quality and diversity of the outputs seems to be the highest with the higher temperatures; the lower temperatures and greedy decoding have repetitive text, whereas the higher temperatures see more coherent output.

Increasing the temperature also increased the diversity of the response: lower temperatures seem to gravitate towards the U.S., whereas the higher temperatures explored other topics (both $T=1.2$ and $T=1.5$ completed with narratives).